In [1]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra import OperationTimedOut

# Configuração de autenticação e conexão
auth_provider = PlainTextAuthProvider(username='cassandra', password='cassandra')
cluster = Cluster(['cassandra'], port=9042, auth_provider=auth_provider)

try:
    # Inicia a sessão de conexão
    session = cluster.connect()

    # Verifica se a conexão foi bem-sucedida executando uma consulta simples
    session.execute("SELECT * FROM system.local")
    print("Conexão estabelecida com sucesso!")

except OperationTimedOut:
    print("Falha na conexão ao servidor Cassandra")


Conexão estabelecida com sucesso!


In [2]:
import pandas as pd
import numpy as np
import uuid

# Lê o CSV original, tolerando linhas malformadas
df = pd.read_csv(
    'MICRODADOS_ED_SUP_IES_2022_corrigido_UTF8.csv',
    encoding='utf-8',
    engine='python',
    on_bad_lines='skip'
)

# Normaliza nomes de colunas
df.columns = [c.lower() for c in df.columns]

# Colunas que devem ser texto (mesmo que pareçam número)
colunas_text = {
    'no_regiao_ies', 'no_uf_ies', 'sg_uf_ies', 'no_municipio_ies',
    'no_mesorregiao_ies', 'no_microrregiao_ies', 'no_mantenedora',
    'no_ies', 'sg_ies', 'ds_endereco_ies', 'ds_numero_endereco_ies',
    'ds_complemento_endereco_ies', 'no_bairro_ies', 'nu_cep_ies'
}

# Corrige colunas de texto que vieram como float (ex: CEP 75132400.0 -> "75132400")
for col in colunas_text:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda v: str(int(v)) if isinstance(v, float) and not pd.isna(v) and float(v).is_integer()
            else (str(v) if pd.notna(v) else None)
        )

# Adiciona a coluna id como primeira coluna, com UUID gerado
df.insert(0, 'id', [str(uuid.uuid4()) for _ in range(len(df))])

# Salva o CSV corrigido, com todos os campos de texto entre aspas
df.to_csv(
    'ies_corrigido.csv',
    index=False,
    encoding='utf-8',
    quoting=1  # csv.QUOTE_ALL -> coloca aspas em TODOS os campos, evita qualquer problema de vírgula
)

print(f"Arquivo gerado com {len(df)} linhas.")

Arquivo gerado com 2289 linhas.


In [3]:
session.execute("""
CREATE KEYSPACE IF NOT EXISTS AulaDemo 
WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};
""")

In [4]:
session.set_keyspace('aulademo')

In [5]:
linhas = session.execute("SELECT keyspace_name FROM system_schema.keyspaces")

for linha in linhas:
    print(linha.keyspace_name)

system_auth
system_schema
inep
aulademo
system_distributed
system
system_traces
testepersistencia


In [6]:
session.execute("""CREATE TABLE IF NOT EXISTS Estudantes (
    id UUID PRIMARY KEY,
    nome TEXT,
    idade INT,
    curso TEXT,
    email TEXT
);""")



In [7]:
tabelas = session.execute("SELECT table_name FROM system_schema.tables WHERE keyspace_name = 'aulademo';")

for tabela in tabelas:
    print(tabela.table_name)

estudantes


In [8]:
session.execute("INSERT INTO Estudantes (id, nome, idade, curso, email) VALUES (uuid(), 'Domitila Canto', 22, 'Letras', 'domitila.canto@email.com');")
session.execute("INSERT INTO Estudantes (id, nome, idade, curso, email) VALUES (uuid(), 'João Leite', 22, 'Engenharia da Computação', 'joao.leite@email.com');")
session.execute("INSERT INTO Estudantes (id, nome, idade, curso, email) VALUES (uuid(), 'Daniel Dias', 25, 'Direito', 'daniel@email.com');")
session.execute("INSERT INTO Estudantes (id, nome, idade, curso, email) VALUES (uuid(), 'Carlos Silva', 23, 'Sistemas', 'carlos@email.com');")
session.execute("INSERT INTO Estudantes (id, nome, idade, curso, email) VALUES (uuid(), 'Eduarda Alves', 19, 'Medicina', 'eduarda@email.com');")

In [9]:
query = session.execute("SELECT * FROM estudantes;")

for estudante in query:
    print(estudante)

Row(id=UUID('de0c2776-c793-4ee0-b048-a9a3d9c47acb'), curso='Engenharia da Computação', email='joao.leite@email.com', idade=22, nome='João Leite')
Row(id=UUID('e23ea57e-c1c2-4082-9859-686c56d42af3'), curso='Sistemas', email='carlos@email.com', idade=23, nome='Carlos Silva')
Row(id=UUID('1cd8a4d0-16b5-4c73-a62f-09b8b0948431'), curso='Medicina', email='eduarda@email.com', idade=19, nome='Eduarda Alves')
Row(id=UUID('1a2d3334-be4a-46ee-86e5-8908f2fb53a2'), curso='Direito', email='daniel@email.com', idade=25, nome='Daniel Dias')
Row(id=UUID('f8fc5da1-7750-4ff6-b741-0f6b4a86b722'), curso='Letras', email='domitila.canto@email.com', idade=22, nome='Domitila Canto')


In [10]:
def altera_estudante_por_nome(nome, operacao, campo_update=None, valor_update=None):
    query_select = f"SELECT id FROM Estudantes WHERE nome = '{nome}' ALLOW FILTERING"
    resultado = session.execute(query_select)
    registros = list(resultado)
    
    if not registros:
        print(f"Estudante '{nome}' não encontrado.")
        return
        
    id_estudante = registros[0].id
    
    if operacao == 'DELETE':
        session.execute(f"DELETE FROM Estudantes WHERE id = {id_estudante}")
        
    elif operacao == 'UPDATE' and campo_update and valor_update:
        valor_formatado = f"'{valor_update}'" if isinstance(valor_update, str) else valor_update
        query_update = f"UPDATE Estudantes SET {campo_update} = {valor_formatado} WHERE id = {id_estudante}"
        session.execute(query_update)

print("Função carregada na memória.")

Função carregada na memória.


In [11]:
altera_estudante_por_nome('Eduarda Alves', 'UPDATE', campo_update='idade', valor_update=99)
altera_estudante_por_nome('Carlos Silva', 'UPDATE', campo_update='curso', valor_update='Filosofia')
altera_estudante_por_nome('Daniel Dias', 'DELETE')

In [12]:
query = session.execute("SELECT * FROM estudantes;")

for estudante in query:
    print(estudante)

Row(id=UUID('de0c2776-c793-4ee0-b048-a9a3d9c47acb'), curso='Engenharia da Computação', email='joao.leite@email.com', idade=22, nome='João Leite')
Row(id=UUID('e23ea57e-c1c2-4082-9859-686c56d42af3'), curso='Filosofia', email='carlos@email.com', idade=23, nome='Carlos Silva')
Row(id=UUID('1cd8a4d0-16b5-4c73-a62f-09b8b0948431'), curso='Medicina', email='eduarda@email.com', idade=99, nome='Eduarda Alves')
Row(id=UUID('f8fc5da1-7750-4ff6-b741-0f6b4a86b722'), curso='Letras', email='domitila.canto@email.com', idade=22, nome='Domitila Canto')


In [13]:
session.execute("DROP KEYSPACE aulademo")
session.shutdown()
cluster.shutdown()